# Run the ADA2 LLM conditions (team notebook)

Use this notebook to run one or more conditions (A0 to A4) against the model and read what comes back. Each run is saved with its exact input, settings and output, so anyone can check it later.

**Before you start**
1. Put your OpenAI key in a file named `.env` in this folder, as one line: `OPENAI_API_KEY=...`. Never share or commit that file.
2. Do not edit the files in `prompts/`, `context/` or `evidence/` here. Those are shared and must stay identical for everyone. Propose changes to the team instead.

**Rules this notebook follows**
- It runs in **pilot** mode only. Pilot runs are rehearsals and must **not** be counted as study data. The formal batch is a separate step, done after the team freezes the prompt (see the last cell).
- Runs use the **real model** (`USE_REAL_MODEL = True` in section 1) and cost money. Set it to `False` only to test the notebook with a fake model; fake runs are saved separately and are not results.
- Nothing here says what the model's answer means biologically. The hypotheses are suggestions to review, not findings.

## 0. Setup (run once)

In [ ]:
import sys, json, csv
from pathlib import Path
from datetime import datetime, timezone
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "runner"))
import importlib, llm_runner as L
importlib.reload(L)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
cfg = L.load_config(ROOT / "config.json")
print("model:", cfg["model"], "| prompt version:", cfg["prompt_version"], "| frozen:", cfg["frozen"])
print("conditions:", {c: (k or ["no evidence"]) for c, k in cfg["conditions"].items()})

## 1. Your run settings
Edit this cell only.

In [ ]:
USE_REAL_MODEL = True           # True = real API call (costs money). False = free fake model, only for testing the notebook.
CONDITIONS = ["A0", "A3"]       # any of A0, A1, A2, A3, A4. A1 and A2 are still placeholders (see section 2).
REPEATS = 1                     # how many times to run each condition
EFFORT = None                   # None = use config.json (currently medium). Or "low" / "medium" / "high".
RUN_BY = "your name"            # recorded in your summary file

## 2. Check the inputs (free)
All conditions must share the same context and system prompt, and each evidence package must appear unchanged inside A4. A condition marked PLACEHOLDER has no real evidence yet; it can be rehearsed but is not meaningful for results.

In [ ]:
if EFFORT: cfg["reasoning_effort"] = EFFORT
checks = L.consistency_checks(cfg)
for name, ok, note in checks:
    print("PASS" if ok else "FAIL", name, f"[{note}]")
rows = []
for c in CONDITIONS:
    p = L.build_prompt(cfg, c)
    rows.append(dict(condition=c, evidence=cfg["conditions"][c], approx_input_tokens=p["input_chars"] // 4,
                     placeholders=p["placeholders"], input_hash=p["input_sha256"][:12]))
display(pd.DataFrame(rows))
assert all(ok for _, ok, _ in checks), "A check failed. Do not run until it is fixed; tell the team."
if any(r["placeholders"] for r in rows):
    print("NOTE: a chosen condition has a PLACEHOLDER evidence package. Treat its output as a rehearsal only.")

## 3. Run
Runs the conditions in a random order (seed from `config.json`) and saves each to `runs_pilot/` (or `runs_mock/` when the fake model is on).

In [ ]:
MODE_DIR = "pilot" if USE_REAL_MODEL else "mock"
if USE_REAL_MODEL:
    print(f"Real run: {len(CONDITIONS) * REPEATS} call(s) to {cfg['model']} (effort: {cfg['reasoning_effort']}). This costs money.")
rows = L.run_batch(cfg, CONDITIONS, repeats=REPEATS, mode="pilot", mock=not USE_REAL_MODEL)
SESSION = pd.DataFrame(rows)
cols = ["run_id", "condition", "model_returned", "status", "n_hypotheses", "input_tokens", "output_tokens", "reasoning_tokens", "unknown_ids", "error"]
display(SESSION[[c for c in cols if c in SESSION.columns]])

**What to check:** `status` should be `completed`. `format_problem`, `parse_error` or `error` means something went wrong (the `error` column says what). `unknown_ids` should be empty: any ID listed there was cited by the model but never given to it.

## 4. Read the hypotheses (this session's runs only)

In [ ]:
mine = set(SESSION.run_id)
for r in L.load_runs(MODE_DIR):
    if r["meta"]["run_id"] not in mine: continue
    print("=" * 100, "\n", r["meta"]["run_id"], "| condition", r["meta"]["condition"], "|", r["meta"]["model_returned"], "|", r["meta"]["status"])
    for i, h in enumerate(r["output"]["hypotheses"], 1):
        print(f"\n  H{i}: {h['title']}  (model's confidence: {h['confidence']})")
        print("   Explanation:", h["question_and_explanation"])
        print("   Cites:", h["supporting_evidence_ids"], "| conflicts with:", h["conflicting_evidence_ids"])
        print("   Test:", h["proposed_test"])
        print("   Would weaken it:", h["falsifying_outcome"])
        print("   Controls and feasibility:", h["controls_and_feasibility"])
        print("   What the computation added:", h["added_value_from_computation"])
    print("\n  Overall note:", r["output"].get("overall_note"))

**How to read this:** the model's confidence is its own label, not a measure of being right. Check that each cited record says what the model claims it says, and that no result is stated that the evidence does not give.

## 5. Citation check

In [ ]:
rep = pd.DataFrame(L.citation_report(MODE_DIR))
rep = rep[rep.run_id.isin(mine)] if len(rep) else rep
display(rep[["run_id", "condition", "hypothesis", "n_cited", "unknown_ids", "title"]] if len(rep) else "no outputs")

**What to look for:** an empty `unknown_ids` means every cited ID was in the model's input. It does not prove the model described each record correctly; read the text against the evidence. In A0 the model should cite only background (`CTX-`) records, since it has no evidence.

## 6. Save a summary of this session

In [ ]:
out = ROOT / f"runs_{MODE_DIR}" / f"session_summary_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.csv"
summ = SESSION[[c for c in cols if c in SESSION.columns]].copy()
summ.insert(0, "run_by", RUN_BY)
summ.insert(1, "real_model", USE_REAL_MODEL)
summ.to_csv(out, index=False)
print("saved", out.relative_to(ROOT))

## Formal runs are not done here
The formal batch (5 conditions x 3 repeats) is blocked by the runner until `config.json` has `"frozen": true`, a final (non-DRAFT) prompt version and no placeholder evidence. When the team has agreed those, it is run from the command line:

`python runner/llm_runner.py run --formal`

Then `python runner/llm_runner.py check --mode formal` and `python runner/llm_runner.py review-sheet --mode formal` for the citation check and the blind review sheet (keep the key file away from reviewers).